In [0]:
#Cargando funciones
import sys
import matplotlib.pyplot as plt

# La raíz del proyecto debe estar en sys.path para que el paquete src
# (carpeta con módulos .py) sea importable desde este notebook.

from src.data_loader import load_dataset
from src.data_validator import validate_dataset
from src.data_cleaner import clean_sales_data
from src.data_analyzer import (
    analyze_products,
    analyze_customers
)

In [0]:
#Carga de datasets       
sales = load_dataset(
    "data/raw/Sales_dataset.csv",
    dtype={
        "cd_NumParte": "string",
        "cd_IdClienteFC": "string"
    }
)

inventory = load_dataset(
    "data/raw/Inventory_dataset.csv",
    dtype={
        "cd_LocId": "string",
        "cd_Id": "string",
        "cd_NumParte": "string"
    }
)
#

In [0]:
#Validando calidad

sales_required = [
    "cd_NumParte",
    "fh_Fecha",
    "ct_Cantidad",
    "im_PrecioVenta"
]

quality = validate_dataset(
    sales,
    sales_required
)

display(quality)

In [0]:
#Limpieza de datos
sales_clean = clean_sales_data(
    sales
)

display(
    sales_clean[
        [
            "cd_NumParte",
            "fh_Fecha",
            "ct_Cantidad",
            "im_PrecioVenta",
            "fl_ParteValida",
            "fl_CantidadValida",
            "fl_PrecioValido",
            "fl_FechaValida",
            "fl_RegistroValido"
        ]
    ].head(20)
)


In [0]:
#Contar registros válidos
display(
    sales_clean[
        "fl_RegistroValido"
    ].value_counts()
)

valid_percentage = (
    sales_clean["fl_RegistroValido"]
    .mean()
    * 100
)

print(
    f"\nPorcentaje de registros válidos: "
    f"{valid_percentage:.2f}%"
)

In [0]:
#Revisar porqué algunos registros no son válidos
print(
    "Número de parte inválido:",
    (~sales_clean["fl_ParteValida"]).sum()
)

print(
    "Cantidad inválida:",
    (~sales_clean["fl_CantidadValida"]).sum()
)

print(
    "Precio inválido:",
    (~sales_clean["fl_PrecioValido"]).sum()
)

print(
    "Fecha inválida:",
    (~sales_clean["fl_FechaValida"]).sum()
)

In [0]:
#Crear el dataset para la siguiente etapa
sales_analysis = (
    sales_clean[
        sales_clean["fl_RegistroValido"]
    ]
    .copy()
)

#Comprobamos
print(
    "Registros originales:",
    len(sales)
)

print(
    "Registros después de limpieza:",
    len(sales_analysis)
)

print(
    "Registros que requieren revisión:",
    len(sales_clean) - len(sales_analysis)
)

**Conclusión**
La separación es justamente la ventaja del framework:

data_loader.py → carga.

data_validator.py → diagnostica.

data_cleaner.py → estandariza y clasifica.

sales_analysis → queda listo para integración/análisis.

In [0]:
#Probando funciones de analisis de productos
product_analysis = analyze_products(sales)

display(
    product_analysis.head(10)
)





In [0]:
#Prueba visual rápida de analisis de clientes
top_products = product_analysis.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_products["cd_NumParte"],
    top_products["cantidad_vendida"]
)

plt.title(
    "Top 10 productos por cantidad vendida"
)

plt.xlabel("Cantidad vendida")
plt.ylabel("Número de parte")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
#Probando análisis de clientes
customer_analysis = analyze_customers(sales)

display(
    customer_analysis.head(10)
)

In [0]:
#Consulta del cliente mayor
top_customer = (
    customer_analysis.iloc[0]
)

print(
    "Cliente con mayor valor de compra:",
    top_customer["cd_IdClienteFC"]
)

print(
    "Cantidad comprada:",
    top_customer["cantidad_comprada"]
)

print(
    "Importe comprado:",
    top_customer["importe_comprado"]
)

print(
    "Productos distintos:",
    top_customer["productos_distintos"]
)

In [0]:
#Visualización de clientes
top_customers = (
    customer_analysis.head(10)
)

plt.figure(figsize=(10, 6))


plt.barh(
    top_customers["cd_IdClienteFC"],
    top_customers["importe_comprado"]
)

plt.title(
    "Top 10 clientes por importe comprado"
)

plt.xlabel("Importe comprado")
plt.ylabel("Cliente")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()